# Preamble

In [1]:
from src.parameter_finder import pulse_parameter_finder, add_buffer_levels, \
                                 construct_U_realized, display_params
import numpy as np
from numpy import pi, sqrt, exp
# np.set_printoptions(precision=5)

In [2]:
def overlap_unitary_fidelity(Uid, U):
    d = Uid.shape[0]
    try:
        Uid, U = Uid.full(), U.full()
    except:
        pass
    return (np.abs(np.trace(Uid.conj().T @ U)))/d

def overlap_state_fidelity(psi_id, psi):
    try:
        fid = fidelity(psi_id, psi)
    except:
        fid = np.abs(psi_id.conj().T @ psi)[0][0]
    return fid**2

def chop(x, delta=1e-4):
    '''Replace numbers<delta with 0'''
    x = np.array(x)
    if x.dtype == 'int32' or x.dtype == 'float64':
        x[abs(x) < delta] = 0
    if x.dtype == 'complex128':
        x.real[abs(x.real) < delta] = 0
        x.imag[abs(x.imag) < delta] = 0
    else: print('Operation incompatible')
    return x
    
def hadamard(d):
    '''Qudit Hadamard gate'''
    w = exp(2j*pi/d)
    mat = np.ones([d,d], dtype=complex)
    for ii in np.arange(1,d):
        for jj in np.arange(1,d):
            mat[ii,jj] = w**(ii*jj)
    return mat/sqrt(d)

# Select a gate to decompose (and its dimension size)

In [3]:
d_gate = 2
qudit_gate = hadamard(d_gate)

# Displacement gates move the state to higher dimensions, so need to define number of bumper states
d_cut = 12 # cutoff dimension = qudit_dim + buffer_levels
target_cavity_gate = add_buffer_levels(qudit_gate, d_cut) # adding buffer levels

print("qudit_gate:")
print(qudit_gate)

qudit_gate:
[[ 0.70710678+0.00000000e+00j  0.70710678+0.00000000e+00j]
 [ 0.70710678+0.00000000e+00j -0.70710678+8.65956056e-17j]]


# Decompose the gate into Snap and Displacement 
Runs an optimizer to solve for parameters

In [4]:
d_snap, n_snap = 3, 2 # snap action dimension, no. of (multi-parameter) snap pulses

use_guess = True # If True, set displacement & phase values below in 'guess'
if use_guess:
    guess = np.concatenate(([-0.5,0.8,0.5], (np.pi) * np.random.randint(2, size=n_snap*d_snap))) 
    # displacement array length should be (n_snap + 1) if use_guess = 1
    output = pulse_parameter_finder(target_cavity_gate, n_snap, d_cut, n_levels=d_snap, max_runs=50,\
                                    err_th=0.01, d_fid=10, initial_guess = guess)
else:
    output = pulse_parameter_finder(target_cavity_gate, n_snap, d_cut, n_levels=d_snap, max_runs=50,\
                                    err_th=0.02, d_fid=10, initial_guess = None)
    # Optimizer stops if infidelity < err_th
    # d_cut >= d_fid >= qudit dimension; d_fid is used to compute unitary fidelity
display_params(output[1], n_snap)
print(f"Fidelity, infidelity (d=d_fid): {round(1-output[3],6)} , {round(output[3],6)}")

Current itr#: 0
Current itr#: 10
Displacements: [-0.35130098  0.32486913  0.02842944]
SNAP degrees:
[[213.4  22.2  13. ]
 [147.3 158.  -12.5]]
Fidelity, infidelity (d=d_fid): 0.993902 , 0.006098


## Verify decomposition

In [5]:
# Check fidelity within qudit space
d_qudit, d_cut_new = 3, d_cut + 10
mat = construct_U_realized(output[1], d_cut_new, n_snap)
print(f"Fidelity (cutoff d = {d_cut_new}): {overlap_unitary_fidelity(mat, add_buffer_levels(qudit_gate, d_cut_new)):.6f}")
print(f"Fidelity (qudit d = {d_qudit}): {overlap_unitary_fidelity(mat[:d_qudit,:d_qudit], add_buffer_levels(qudit_gate, d_qudit)):.6f}")

Fidelity (cutoff d = 22): 0.997194
Fidelity (qudit d = 3): 0.982613


In [6]:
# Check argument of the complex unitary
np.rad2deg(np.angle(mat[:d_qudit,:d_qudit]))

array([[   2.19396997,   -5.15295432,   -0.30020748],
       [   4.99005572,  176.41607655, -114.85786323],
       [-155.76123559,  151.09154213,   -1.95794054]])